# Validación MILP de la capa $\theta$

Este notebook valida la formulación MILP exacta de la transformación $\theta$ de Keccak.

La estrategia consiste en:

1. fijar un estado binario de entrada;
2. resolver las restricciones MILP de $\theta$;
3. recuperar la salida obtenida por el solver;
4. calcular la misma transformación con NumPy;
5. comparar ambas salidas posición por posición.

La validación se realiza para:

- estado nulo;
- estado con un solo bit activo;
- estados aleatorios;
- tamaños de palabra $z=4$ y $z=8$.

## 1. Formulación matemática

La paridad de cada columna es:

$$
C[x,k]
=
\bigoplus_{y=0}^{4} A[x,y,k].
$$

Esta operación se representa mediante:

$$
\sum_{y=0}^{4} A[x,y,k]
=
C[x,k]
+
2Q_C[x,k],
$$

donde:

$$
C[x,k]\in\{0,1\},
$$

y:

$$
Q_C[x,k]\in\{0,1,2\}.
$$

El efecto de difusión se define como:

$$
D[x,k]
=
C[x-1 \bmod 5,k]
\oplus
C[x+1 \bmod 5,k-1 \bmod z].
$$

Su linealización es:

$$
C[x-1,k]
+
C[x+1,k-1]
=
D[x,k]
+
2Q_D[x,k],
$$

con:

$$
D[x,k]\in\{0,1\},
$$

y:

$$
Q_D[x,k]\in\{0,1\}.
$$

Finalmente:

$$
T[x,y,k]
=
A[x,y,k]
\oplus
D[x,k].
$$

Se representa mediante:

$$
A[x,y,k]
+
D[x,k]
=
T[x,y,k]
+
2Q_T[x,y,k].
$$

## 2. Importación del proyecto

Se importan:

- la configuración experimental;
- el modelo MILP;
- la implementación funcional de $\theta$;
- utilidades para crear estados y calcular pesos de Hamming.

## 3. Función auxiliar para resolver $\theta$ mediante MILP

La función siguiente:

1. recibe un estado NumPy;
2. crea el modelo MILP;
3. añade la capa $\theta$;
4. fija todos los bits de entrada;
5. define un objetivo de factibilidad;
6. ejecuta CBC;
7. recupera la salida como arreglo NumPy.

El objetivo de factibilidad es constante:

$$
\min 0.
$$

En este caso, la solución queda completamente determinada por las restricciones y por el estado de entrada.

In [2]:
# ============================================================
# CONFIGURACIÓN E IMPORTACIONES
# ============================================================

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd

from keccak_milp import (
    ExperimentConfig,
    KeccakMILPModel,
)
from keccak_milp.layers import (
    create_single_active_bit_state,
    hamming_weight,
    theta,
)


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)


print("=" * 70)
print("ENTORNO DEL NOTEBOOK")
print("=" * 70)
print(f"Raíz del proyecto : {PROJECT_ROOT}")
print(f"Python activo     : {sys.executable}")
print(f"NumPy             : {np.__version__}")
print("=" * 70)

ENTORNO DEL NOTEBOOK
Raíz del proyecto : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Python activo     : d:\venvs\keccak-milp\Scripts\python.exe
NumPy             : 2.5.1


In [3]:
# ============================================================
# FUNCIÓN AUXILIAR DE RESOLUCIÓN
# ============================================================

def resolver_theta_milp(
    estado: np.ndarray,
    limite_tiempo: int = 60,
) -> dict:
    """
    Resuelve la capa theta mediante MILP para un estado fijo.

    Parameters
    ----------
    estado:
        Arreglo binario de forma (5, 5, z).

    limite_tiempo:
        Límite de resolución en segundos.

    Returns
    -------
    dict
        Resultados del modelo y salida MILP.
    """

    if estado.ndim != 3:
        raise ValueError(
            "El estado debe tener forma (5, 5, z)."
        )

    if estado.shape[0] != 5 or estado.shape[1] != 5:
        raise ValueError(
            "El estado debe contener una matriz de 5 x 5 lanes."
        )

    if not np.all(np.isin(estado, [0, 1])):
        raise ValueError(
            "El estado debe ser binario."
        )

    z = estado.shape[2]

    config = ExperimentConfig(
        z=z,
        rounds=1,
        solver="cbc",
        time_limit_seconds=limite_tiempo,
        mip_gap=0.0,
        verbose=False,
    )

    modelo = KeccakMILPModel(config)

    modelo.add_theta_layer(round_index=0)

    modelo.fix_state_values(
        round_index=0,
        values=estado.astype(int).tolist(),
    )

    modelo.set_feasibility_objective()

    inicio = time.perf_counter()
    estado_solver = modelo.solve()
    tiempo = time.perf_counter() - inicio

    salida_milp = np.asarray(
        modelo.theta_output_values(0),
        dtype=np.int64,
    )

    estadisticas = modelo.statistics()

    return {
        "modelo": modelo,
        "estado_solver": estado_solver,
        "tiempo_segundos": tiempo,
        "salida_milp": salida_milp,
        "variables_declaradas": estadisticas.declared_variables,
        "variables_conectadas": estadisticas.attached_variables,
        "restricciones": estadisticas.total_constraints,
    }

## 4. Validación con el estado nulo

El estado nulo se define como:

$$
A[x,y,k]=0
\qquad
\forall x,y,k.
$$

Como todas las paridades son cero:

$$
C[x,k]=0,
$$

y:

$$
D[x,k]=0.
$$

Por tanto:

$$
\theta(A)=A=0.
$$

In [4]:
# ============================================================
# ESTADO NULO PARA z = 4
# ============================================================

estado_nulo_z4 = np.zeros(
    (5, 5, 4),
    dtype=np.int64,
)

salida_numpy_nulo_z4 = theta(
    estado_nulo_z4
)

resultado_nulo_z4 = resolver_theta_milp(
    estado_nulo_z4
)

salida_milp_nulo_z4 = resultado_nulo_z4[
    "salida_milp"
]


print("=" * 70)
print("VALIDACIÓN DEL ESTADO NULO z = 4")
print("=" * 70)
print(f"Estado solver      : {resultado_nulo_z4['estado_solver']}")
print(f"Peso entrada       : {hamming_weight(estado_nulo_z4)}")
print(f"Peso salida NumPy  : {hamming_weight(salida_numpy_nulo_z4)}")
print(f"Peso salida MILP   : {hamming_weight(salida_milp_nulo_z4)}")
print(
    "Coincidencia:",
    np.array_equal(
        salida_numpy_nulo_z4,
        salida_milp_nulo_z4,
    ),
)
print("=" * 70)


assert resultado_nulo_z4["estado_solver"] == "Optimal"
assert np.array_equal(
    salida_numpy_nulo_z4,
    salida_milp_nulo_z4,
)

VALIDACIÓN DEL ESTADO NULO z = 4
Estado solver      : Optimal
Peso entrada       : 0
Peso salida NumPy  : 0
Peso salida MILP   : 0
Coincidencia: True


## 5. Validación con un solo bit activo

Se utiliza:

$$
A[2,3,1]=1.
$$

Todas las demás posiciones permanecen en cero.

La implementación funcional mostró que:

$$
w_H(A)=1,
$$

y:

$$
w_H(\theta(A))=11.
$$

El modelo MILP debe reproducir exactamente el mismo resultado.

In [5]:
# ============================================================
# ESTADO UNITARIO PARA z = 4
# ============================================================

estado_unitario_z4 = create_single_active_bit_state(
    z=4,
    x=2,
    y=3,
    k=1,
)

salida_numpy_unitario_z4 = theta(
    estado_unitario_z4
)

resultado_unitario_z4 = resolver_theta_milp(
    estado_unitario_z4
)

salida_milp_unitario_z4 = resultado_unitario_z4[
    "salida_milp"
]


print("=" * 70)
print("VALIDACIÓN DEL ESTADO UNITARIO z = 4")
print("=" * 70)
print(f"Estado solver      : {resultado_unitario_z4['estado_solver']}")
print(f"Peso entrada       : {hamming_weight(estado_unitario_z4)}")
print(f"Peso salida NumPy  : {hamming_weight(salida_numpy_unitario_z4)}")
print(f"Peso salida MILP   : {hamming_weight(salida_milp_unitario_z4)}")
print(f"Tiempo             : {resultado_unitario_z4['tiempo_segundos']:.6f} s")
print(
    "Coincidencia:",
    np.array_equal(
        salida_numpy_unitario_z4,
        salida_milp_unitario_z4,
    ),
)
print("=" * 70)


assert hamming_weight(salida_milp_unitario_z4) == 11
assert np.array_equal(
    salida_numpy_unitario_z4,
    salida_milp_unitario_z4,
)

VALIDACIÓN DEL ESTADO UNITARIO z = 4
Estado solver      : Optimal
Peso entrada       : 1
Peso salida NumPy  : 11
Peso salida MILP   : 11
Tiempo             : 0.173417 s
Coincidencia: True


## 6. Comparación posición por posición

Para verificar la igualdad completa entre ambas implementaciones, se construye una tabla con:

- coordenadas $(x,y,k)$;
- valor obtenido mediante NumPy;
- valor obtenido mediante MILP;
- indicador de coincidencia.

La igualdad esperada es:

$$
T_{\mathrm{NumPy}}[x,y,k]
=
T_{\mathrm{MILP}}[x,y,k]
$$

para todas las posiciones.

In [6]:
# ============================================================
# COMPARACIÓN POSICIÓN POR POSICIÓN
# ============================================================

comparacion_unitaria = []

for x in range(5):
    for y in range(5):
        for k in range(4):
            valor_numpy = int(
                salida_numpy_unitario_z4[x, y, k]
            )

            valor_milp = int(
                salida_milp_unitario_z4[x, y, k]
            )

            comparacion_unitaria.append(
                {
                    "x": x,
                    "y": y,
                    "k": k,
                    "numpy": valor_numpy,
                    "milp": valor_milp,
                    "coincide": valor_numpy == valor_milp,
                }
            )


df_comparacion_unitaria = pd.DataFrame(
    comparacion_unitaria
)

df_comparacion_unitaria

,x,y,k,numpy,milp,coincide
0,0,0,0,0,0,True
1,0,0,1,0,0,True
2,0,0,2,0,0,True
3,0,0,3,0,0,True
4,0,1,0,0,0,True
...,...,...,...,...,...,...
95,4,3,3,0,0,True
96,4,4,0,0,0,True
97,4,4,1,0,0,True
98,4,4,2,0,0,True


In [7]:
# ============================================================
# CONTROL DE DIFERENCIAS
# ============================================================

diferencias_unitarias = df_comparacion_unitaria[
    ~df_comparacion_unitaria["coincide"]
]

print(f"Número de diferencias: {len(diferencias_unitarias)}")

assert diferencias_unitarias.empty

Número de diferencias: 0


## 7. Inspección de las variables internas

Para $z=4$ y una ronda, el modelo contiene inicialmente dos estados de frontera:

$$
N_{\mathrm{estado}}
=
25(4)(1+1)
=
200.
$$

Las variables de $\theta$ son:

### Paridad de columna

$$
C:
5z=20,
$$

$$
Q_C:
5z=20.
$$

### Efecto de difusión

$$
D:
5z=20,
$$

$$
Q_D:
5z=20.
$$

### Salida de $\theta$

$$
T:
25z=100,
$$

$$
Q_T:
25z=100.
$$

Por tanto:

$$
N_{\theta}
=
20+20+20+20+100+100
=
280.
$$

El total declarado es:

$$
N_{\mathrm{total}}
=
200+280
=
480.
$$

In [8]:
# ============================================================
# INSPECCIÓN DE VARIABLES Y RESTRICCIONES
# ============================================================

modelo_unitario_z4 = resultado_unitario_z4["modelo"]

conteos_theta_z4 = pd.DataFrame(
    [
        {
            "grupo": "Estados de frontera",
            "cantidad": len(modelo_unitario_z4.state),
        },
        {
            "grupo": "Theta C",
            "cantidad": len(modelo_unitario_z4.theta_c),
        },
        {
            "grupo": "Theta QC",
            "cantidad": len(modelo_unitario_z4.theta_qc),
        },
        {
            "grupo": "Theta D",
            "cantidad": len(modelo_unitario_z4.theta_d),
        },
        {
            "grupo": "Theta QD",
            "cantidad": len(modelo_unitario_z4.theta_qd),
        },
        {
            "grupo": "Theta salida T",
            "cantidad": len(modelo_unitario_z4.theta_output),
        },
        {
            "grupo": "Theta QT",
            "cantidad": len(modelo_unitario_z4.theta_qt),
        },
    ]
)

conteos_theta_z4

,grupo,cantidad
0,Estados de frontera,200
1,Theta C,20
2,Theta QC,20
3,Theta D,20
4,Theta QD,20
5,Theta salida T,100
6,Theta QT,100


In [9]:
# ============================================================
# VALIDACIÓN DE CONTEOS PARA z = 4
# ============================================================

print("=" * 70)
print("DIMENSIONES DEL MODELO z = 4")
print("=" * 70)
print(
    f"Variables declaradas : "
    f"{resultado_unitario_z4['variables_declaradas']}"
)
print(
    f"Variables conectadas : "
    f"{resultado_unitario_z4['variables_conectadas']}"
)
print(
    f"Restricciones        : "
    f"{resultado_unitario_z4['restricciones']}"
)
print("=" * 70)


assert resultado_unitario_z4[
    "variables_declaradas"
] == 480

# 140 restricciones de theta + 100 para fijar la entrada.
assert resultado_unitario_z4[
    "restricciones"
] == 240

DIMENSIONES DEL MODELO z = 4
Variables declaradas : 480
Variables conectadas : 381
Restricciones        : 240


## 8. Restricciones del modelo

Las restricciones de $\theta$ para $z=4$ son:

### Paridades $C$

$$
5z=20.
$$

### Efectos $D$

$$
5z=20.
$$

### Salidas $T$

$$
25z=100.
$$

Por tanto:

$$
N_{\theta,\mathrm{restricciones}}
=
20+20+100
=
140.
$$

Para fijar completamente el estado de entrada se agregan:

$$
25z=100
$$

restricciones adicionales.

El total es:

$$
140+100=240.
$$

## 9. Validación con estados aleatorios

Una validación con un único patrón no es suficiente.

Se generarán estados aleatorios y se comprobará:

$$
\theta_{\mathrm{MILP}}(A)
=
\theta_{\mathrm{NumPy}}(A).
$$

Cada estado contendrá una combinación distinta de bits activos, cancelaciones XOR y paridades de columna.

In [10]:
# ============================================================
# VALIDACIÓN ALEATORIA PARA z = 4
# ============================================================

rng = np.random.default_rng(20260712)

resultados_aleatorios_z4 = []

for prueba in range(10):
    estado = rng.integers(
        0,
        2,
        size=(5, 5, 4),
        dtype=np.int64,
    )

    salida_numpy = theta(estado)
    resultado_milp = resolver_theta_milp(estado)
    salida_milp = resultado_milp["salida_milp"]

    coincide = np.array_equal(
        salida_numpy,
        salida_milp,
    )

    resultados_aleatorios_z4.append(
        {
            "prueba": prueba + 1,
            "z": 4,
            "peso_entrada": hamming_weight(estado),
            "peso_salida_numpy": hamming_weight(salida_numpy),
            "peso_salida_milp": hamming_weight(salida_milp),
            "coincide": coincide,
            "estado_solver": resultado_milp["estado_solver"],
            "tiempo_segundos": resultado_milp["tiempo_segundos"],
        }
    )


df_aleatorios_z4 = pd.DataFrame(
    resultados_aleatorios_z4
)

df_aleatorios_z4

,prueba,z,peso_entrada,peso_salida_numpy,peso_salida_milp,coincide,estado_solver,tiempo_segundos
0,1,4,52,48,48,True,Optimal,0.175277
1,2,4,45,45,45,True,Optimal,0.179270
2,3,4,44,52,52,True,Optimal,0.176129
3,4,4,50,52,52,True,Optimal,0.172185
4,5,4,53,59,59,True,Optimal,0.174792
5,6,4,58,42,42,True,Optimal,0.177980
6,7,4,47,39,39,True,Optimal,0.174853
7,8,4,49,53,53,True,Optimal,0.176193
8,9,4,55,47,47,True,Optimal,0.174543
9,10,4,52,50,50,True,Optimal,0.208006


In [11]:
# ============================================================
# CONTROL DE VALIDACIÓN ALEATORIA z = 4
# ============================================================

assert df_aleatorios_z4["coincide"].all()
assert (
    df_aleatorios_z4["estado_solver"]
    == "Optimal"
).all()

print("Las 10 pruebas aleatorias para z=4 fueron validadas.")

Las 10 pruebas aleatorias para z=4 fueron validadas.


## 10. Validación para $z=8$

Para $z=8$, el estado tiene:

$$
25(8)=200
$$

bits.

Los dos estados de frontera aportan:

$$
25(8)(2)=400
$$

variables.

Las variables adicionales de $\theta$ son:

$$
70z=70(8)=560.
$$

Por tanto:

$$
N_{\mathrm{total}}
=
400+560
=
960.
$$

Las restricciones de $\theta$ son:

$$
35z
=
35(8)
=
280.
$$

Al fijar los 200 bits de entrada:

$$
N_{\mathrm{restricciones}}
=
280+200
=
480.
$$

In [12]:
# ============================================================
# ESTADO UNITARIO PARA z = 8
# ============================================================

estado_unitario_z8 = create_single_active_bit_state(
    z=8,
    x=2,
    y=3,
    k=1,
)

salida_numpy_unitario_z8 = theta(
    estado_unitario_z8
)

resultado_unitario_z8 = resolver_theta_milp(
    estado_unitario_z8
)

salida_milp_unitario_z8 = resultado_unitario_z8[
    "salida_milp"
]


print("=" * 70)
print("VALIDACIÓN DEL ESTADO UNITARIO z = 8")
print("=" * 70)
print(f"Estado solver      : {resultado_unitario_z8['estado_solver']}")
print(f"Peso entrada       : {hamming_weight(estado_unitario_z8)}")
print(f"Peso salida NumPy  : {hamming_weight(salida_numpy_unitario_z8)}")
print(f"Peso salida MILP   : {hamming_weight(salida_milp_unitario_z8)}")
print(
    f"Variables          : "
    f"{resultado_unitario_z8['variables_declaradas']}"
)
print(
    f"Restricciones      : "
    f"{resultado_unitario_z8['restricciones']}"
)
print(f"Tiempo             : {resultado_unitario_z8['tiempo_segundos']:.6f} s")
print(
    "Coincidencia:",
    np.array_equal(
        salida_numpy_unitario_z8,
        salida_milp_unitario_z8,
    ),
)
print("=" * 70)


assert np.array_equal(
    salida_numpy_unitario_z8,
    salida_milp_unitario_z8,
)

assert resultado_unitario_z8[
    "variables_declaradas"
] == 960

assert resultado_unitario_z8[
    "restricciones"
] == 480

VALIDACIÓN DEL ESTADO UNITARIO z = 8
Estado solver      : Optimal
Peso entrada       : 1
Peso salida NumPy  : 11
Peso salida MILP   : 11
Variables          : 960
Restricciones      : 480
Tiempo             : 0.186364 s
Coincidencia: True


## 11. Comparación estructural entre $z=4$ y $z=8$

El crecimiento de la capa $\theta$ es lineal respecto a $z$.

La cantidad de variables adicionales es:

$$
N_{\theta,\mathrm{variables}}
=
70z.
$$

La cantidad de restricciones propias de $\theta$ es:

$$
N_{\theta,\mathrm{restricciones}}
=
35z.
$$

Esto significa que duplicar $z$ de 4 a 8 duplica aproximadamente el tamaño estructural de la capa.

In [13]:
# ============================================================
# COMPARACIÓN ESTRUCTURAL
# ============================================================

comparacion_estructural = pd.DataFrame(
    [
        {
            "z": 4,
            "bits_estado": 100,
            "variables_estado": 200,
            "variables_theta": 280,
            "variables_totales": 480,
            "restricciones_theta": 140,
            "restricciones_fijacion": 100,
            "restricciones_totales": 240,
            "tiempo_segundos": resultado_unitario_z4[
                "tiempo_segundos"
            ],
        },
        {
            "z": 8,
            "bits_estado": 200,
            "variables_estado": 400,
            "variables_theta": 560,
            "variables_totales": 960,
            "restricciones_theta": 280,
            "restricciones_fijacion": 200,
            "restricciones_totales": 480,
            "tiempo_segundos": resultado_unitario_z8[
                "tiempo_segundos"
            ],
        },
    ]
)

comparacion_estructural

,z,bits_estado,variables_estado,variables_theta,variables_totales,restricciones_theta,restricciones_fijacion,restricciones_totales,tiempo_segundos
0,4,100,200,280,480,140,100,240,0.173417
1,8,200,400,560,960,280,200,480,0.186364


## 12. Validación aleatoria para $z=8$

Se ejecutarán varias pruebas aleatorias para comprobar que la formulación continúa siendo exacta al duplicar el tamaño de palabra.

In [14]:
# ============================================================
# VALIDACIÓN ALEATORIA PARA z = 8
# ============================================================

rng_z8 = np.random.default_rng(20260808)

resultados_aleatorios_z8 = []

for prueba in range(5):
    estado = rng_z8.integers(
        0,
        2,
        size=(5, 5, 8),
        dtype=np.int64,
    )

    salida_numpy = theta(estado)
    resultado_milp = resolver_theta_milp(estado)
    salida_milp = resultado_milp["salida_milp"]

    resultados_aleatorios_z8.append(
        {
            "prueba": prueba + 1,
            "z": 8,
            "peso_entrada": hamming_weight(estado),
            "peso_salida_numpy": hamming_weight(salida_numpy),
            "peso_salida_milp": hamming_weight(salida_milp),
            "coincide": np.array_equal(
                salida_numpy,
                salida_milp,
            ),
            "estado_solver": resultado_milp["estado_solver"],
            "tiempo_segundos": resultado_milp["tiempo_segundos"],
        }
    )


df_aleatorios_z8 = pd.DataFrame(
    resultados_aleatorios_z8
)

df_aleatorios_z8

,prueba,z,peso_entrada,peso_salida_numpy,peso_salida_milp,coincide,estado_solver,tiempo_segundos
0,1,8,107,111,111,True,Optimal,0.191993
1,2,8,110,106,106,True,Optimal,0.205650
2,3,8,102,112,112,True,Optimal,0.195144
3,4,8,96,110,110,True,Optimal,0.192700
4,5,8,107,99,99,True,Optimal,0.198433


In [15]:
# ============================================================
# CONTROL DE VALIDACIÓN ALEATORIA z = 8
# ============================================================

assert df_aleatorios_z8["coincide"].all()
assert (
    df_aleatorios_z8["estado_solver"]
    == "Optimal"
).all()

print("Las pruebas aleatorias para z=8 fueron validadas.")

Las pruebas aleatorias para z=8 fueron validadas.


## 13. Consolidación de resultados

Los resultados de las pruebas aleatorias para ambos tamaños se consolidan en una sola tabla.

La columna `coincide` debe ser verdadera en todos los casos.

In [17]:
# ============================================================
# CONSOLIDACIÓN
# ============================================================

df_validacion_theta_milp = pd.concat(
    [
        df_aleatorios_z4,
        df_aleatorios_z8,
    ],
    ignore_index=True,
)

df_validacion_theta_milp

,prueba,z,peso_entrada,peso_salida_numpy,peso_salida_milp,coincide,estado_solver,tiempo_segundos
0,1,4,52,48,48,True,Optimal,0.175277
1,2,4,45,45,45,True,Optimal,0.179270
2,3,4,44,52,52,True,Optimal,0.176129
3,4,4,50,52,52,True,Optimal,0.172185
4,5,4,53,59,59,True,Optimal,0.174792
5,6,4,58,42,42,True,Optimal,0.177980
6,7,4,47,39,39,True,Optimal,0.174853
7,8,4,49,53,53,True,Optimal,0.176193
8,9,4,55,47,47,True,Optimal,0.174543
9,10,4,52,50,50,True,Optimal,0.208006


In [18]:
# ============================================================
# RESUMEN POR TAMAÑO z
# ============================================================

df_resumen_theta = (
    df_validacion_theta_milp
    .groupby("z")
    .agg(
        pruebas=("prueba", "count"),
        coincidencias=("coincide", "sum"),
        tiempo_promedio_segundos=(
            "tiempo_segundos",
            "mean",
        ),
        tiempo_maximo_segundos=(
            "tiempo_segundos",
            "max",
        ),
    )
    .reset_index()
)

df_resumen_theta

,z,pruebas,coincidencias,tiempo_promedio_segundos,tiempo_maximo_segundos
0,4,10,10,0.178923,0.208006
1,8,5,5,0.196784,0.205650


## 14. Exportación de resultados

Se guardarán dos archivos:

```text
results/tables/validacion_theta_milp.csv
```
y
```text
results/tables/resumen_theta_milp.csv
```
El primer archivo contiene cada ejecución individual.

El segundo resume:

cantidad de pruebas;
coincidencias;
tiempo promedio;
tiempo máximo.

In [19]:
# ============================================================
# EXPORTACIÓN
# ============================================================

ruta_validacion = (
    TABLES_DIR / "validacion_theta_milp.csv"
)

ruta_resumen = (
    TABLES_DIR / "resumen_theta_milp.csv"
)

df_validacion_theta_milp.to_csv(
    ruta_validacion,
    index=False,
    encoding="utf-8-sig",
)

df_resumen_theta.to_csv(
    ruta_resumen,
    index=False,
    encoding="utf-8-sig",
)


print("=" * 70)
print("ARCHIVOS GENERADOS")
print("=" * 70)
print(ruta_validacion)
print(ruta_resumen)
print("=" * 70)


assert ruta_validacion.exists()
assert ruta_resumen.exists()

ARCHIVOS GENERADOS
d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\results\tables\validacion_theta_milp.csv
d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\results\tables\resumen_theta_milp.csv


## 15. Interpretación computacional

La formulación MILP de $\theta$ es exacta, pero introduce variables auxiliares.

Por ronda:

$$
N_{\theta,\mathrm{variables}}
=
70z.
$$

Y:

$$
N_{\theta,\mathrm{restricciones}}
=
35z.
$$

Para $z=4$:

$$
N_{\theta,\mathrm{variables}}=280,
$$

$$
N_{\theta,\mathrm{restricciones}}=140.
$$

Para $z=8$:

$$
N_{\theta,\mathrm{variables}}=560,
$$

$$
N_{\theta,\mathrm{restricciones}}=280.
$$

El crecimiento es lineal respecto a $z$.

Sin embargo, el modelo completo crecerá también con el número de rondas:

$$
N_{\theta,\mathrm{variables,total}}
=
70zR.
$$

Por ejemplo, para $z=8$ y tres rondas:

$$
70(8)(3)=1680
$$

variables asociadas únicamente a $\theta$.

## 16. Conclusiones

Se comprobó que:

1. la formulación MILP representa exactamente las operaciones XOR de $\theta$;
2. la salida MILP coincide con NumPy para estados nulos;
3. la salida coincide para estados con un único bit activo;
4. la salida coincide para estados aleatorios;
5. la formulación funciona para $z=4$ y $z=8$;
6. el crecimiento de variables y restricciones es lineal respecto a $z$;
7. CBC resuelve estas instancias sin dificultad.

La siguiente etapa será conectar la salida de $\theta$ con las capas $\rho$ y $\pi$ dentro del modelo MILP.

Como $\rho$ y $\pi$ son permutaciones, no requieren variables auxiliares de paridad. Bastará con establecer igualdades o reutilizar directamente la correspondencia de índices.

In [20]:
# ============================================================
# CONTROL FINAL
# ============================================================

assert np.array_equal(
    salida_numpy_nulo_z4,
    salida_milp_nulo_z4,
)

assert np.array_equal(
    salida_numpy_unitario_z4,
    salida_milp_unitario_z4,
)

assert np.array_equal(
    salida_numpy_unitario_z8,
    salida_milp_unitario_z8,
)

assert df_validacion_theta_milp["coincide"].all()

assert (
    df_validacion_theta_milp["estado_solver"]
    == "Optimal"
).all()

assert ruta_validacion.exists()
assert ruta_resumen.exists()


print("=" * 70)
print("FORMULACIÓN MILP DE THETA VALIDADA")
print("=" * 70)
print("El proyecto está listo para integrar rho y pi al modelo MILP.")
print("=" * 70)

FORMULACIÓN MILP DE THETA VALIDADA
El proyecto está listo para integrar rho y pi al modelo MILP.
